# 01 · Análisis Exploratorio y Preparación de Datos
**Notebook común del equipo** (Alejandro Flores · Paul Rosero · Gloria Chushig)

## 1. Contexto y pregunta de negocio
**¿Qué predice el modelo?** Si una URL es legítima (benigna, `0`) o maliciosa (phishing, `1`).

**¿Por qué importa?** El phishing es uno de los ataques más comunes. Advertir al usuario antes de abrir
un enlace malicioso previene el robo de credenciales y pérdidas económicas.

In [ ]:
# Configuración: permite importar el código común de src/ desde la carpeta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
plt.style.use("ggplot")
sns.set_palette("muted")

from src import config, data, evaluation

## 2. Carga y estructura inicial

In [ ]:
df = data.cargar_dataset()   # lee data/Dataset.csv e imputa 'tld' nulo con 'none'
display(df.head())
print("Forma del dataset:", df.shape)
df.info()

In [ ]:
# Columnas reales del dataset → con esto se llena config.FEATURES y se alinea el extractor
print(list(df.columns))

## 3. Valores nulos

In [ ]:
crudo = pd.read_csv(config.RUTA_DATASET)
nulos = crudo.isnull().sum()
display(nulos[nulos > 0])

Los nulos de `tld` corresponden a URLs que son una **IP pura** (p. ej. `http://195.191.230.50/...`),
que por definición no tienen dominio de nivel superior. Se imputan con `'none'` en `src/data.py`.
Además, `tld` es texto y **no entra al modelo** (ver `config.COLUMNAS_EXCLUIDAS`).

## 4. Distribución de la variable objetivo

In [ ]:
conteo = df[config.COLUMNA_OBJETIVO].value_counts()
dist = pd.DataFrame({"Cantidad": conteo, "Porcentaje (%)": (conteo / conteo.sum() * 100).round(2)})
display(dist)

In [ ]:
# Gráfico 1: distribución de clases
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x=config.COLUMNA_OBJETIVO, ax=ax)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}\n({p.get_height() / len(df):.1%})",
                (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
ax.set_title("Distribución de clases (0: Benigna, 1: Phishing)")
plt.show()

**Interpretación:** TODO(equipo) — explicar el desbalance (~86% / ~14%), por qué refleja la realidad
y por qué obliga a usar `stratify=y` y métricas como Recall, F1 y AUC en lugar de solo Accuracy.

## 5. Correlación de variables con la etiqueta

In [ ]:
features = data.obtener_features(df)
print(f"{len(features)} variables para el modelo:", features)

# Gráfico 2: heatmap de correlación
# TODO(equipo): elegir las variables más relevantes si el mapa queda muy cargado.
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df[features + [config.COLUMNA_OBJETIVO]].corr(), cmap="coolwarm", center=0,
            annot=len(features) <= 12, fmt=".2f", ax=ax)
ax.set_title("Matriz de correlación")
plt.show()

**Interpretación:** TODO(equipo).

## 6. Comparación de variables por clase

In [ ]:
# Gráfico 3: boxplots / histogramas segmentados por clase
# TODO(equipo): elegir las variables (p. ej. longitud de URL y proporción de dígitos).
variables_a_comparar = []   # ej: ["url_len", "digit_ratio"]

for col in variables_a_comparar:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.boxplot(data=df, x=config.COLUMNA_OBJETIVO, y=col, ax=ax[0], showfliers=False)
    sns.histplot(data=df, x=col, hue=config.COLUMNA_OBJETIVO, stat="density",
                 common_norm=False, bins=50, ax=ax[1])
    fig.suptitle(f"{col} por clase")
    plt.show()

**Interpretación:** TODO(equipo).

## 7. División y preprocesamiento (común a los 3 modelos)
La partición estratificada 80/20 y el `ColumnTransformer` están en `src/data.py` y
`src/preprocessing.py`, así **los tres modelos usan exactamente los mismos datos**.

In [ ]:
X_train, X_test, y_train, y_test = data.dividir(df)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("% phishing train:", round(y_train.mean() * 100, 2), "| test:", round(y_test.mean() * 100, 2))

## 8. Validación del extractor
Antes de entrenar hay que confirmar que `src/extractor.py` calcula cada variable igual que el dataset;
si no, la app web dará predicciones incorrectas.

In [ ]:
from scripts.validar_extractor import main as validar_extractor
validar_extractor(200)